# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. You will learn to inspect metadata, enumerate record sets, load records, and perform basic analysis and visualizations.

### Dataset Source
This dataset is defined by a Croissant schema and referenced by a schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and explore available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print("Dataset Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published (date):", getattr(dataset.metadata, 'datePublished', 'N/A'))
print("Identifier:", getattr(dataset.metadata, 'identifier', ''))
print("Authors:")
for author in getattr(dataset.metadata, 'author', []):
    print("-", getattr(author, 'name', getattr(author, '@id', 'N/A')))

## 2. Data Overview
List available record sets (tables) and their fields, each by their Croissant `@id`.

Record sets are the main tabular resources in the Croissant schema. Fields/columns are listed per record set and referenced by their `@id`.

In [ ]:
# List all record sets with their @id and label, and corresponding fields
record_sets = dataset.metadata.recordSets
print("Available Record Sets (by @id):")
for rs in record_sets:
    label = getattr(rs, 'name', getattr(rs, '@id', ''))
    print(f"- Record Set @id: {rs['@id']} | Name: {label}")
    if hasattr(rs, 'fields') or 'fields' in rs:
        print("  Fields (by @id):")
        for field in rs['fields']:
            fname = getattr(field, 'name', getattr(field, '@id', ''))
            print(f"    - {field['@id']} | Name: {fname}")
    elif hasattr(rs, 'columns') or 'columns' in rs:
        print("  Columns (by @id):")
        for col in rs['columns']:
            cname = getattr(col, 'name', getattr(col, '@id', ''))
            print(f"    - {col['@id']} | Name: {cname}")
    print()

## 3. Data Extraction
Extract records from a specific record set into a Pandas DataFrame for further analysis. All operations reference record sets and fields by their Croissant `@id`.

In [ ]:
# Gather the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print("Record sets available for extraction:")
print(record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting records from {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records, columns: {df.columns.tolist()}")
    except Exception as e:
        print("Failed to load records:", e)
# Choose one for demonstration:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print("\nFirst 5 rows of records from record set:", main_record_set_id)
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Perform common operations such as filtering numeric fields, normalizing, and grouping -- all using fields by their `@id`.

For demonstration, we select a numerical field in the main record set.

In [ ]:
# Inspect columns for likely numeric fields
df = dataframes[main_record_set_id]
print("Columns available:", df.columns.tolist())

# Example: Find the first float/integer-like field
numeric_field = None
for col in df.columns:
    # Try to check data type by sampling values
    sample = df[col].dropna()
    if sample.empty:
        continue
    try:
        sample = pd.to_numeric(sample)
        numeric_field = col
        break
    except Exception:
        continue
if numeric_field is None:
    print("No numeric field found; cannot proceed with numeric EDA.")
else:
    print(f"Numeric field for analysis (by @id): {numeric_field}")
    # Filter: keep only > 10
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (count: {filtered_df.shape[0]}):")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (z-score):")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by first string/categorical field
    group_field = None
    for col in df.columns:
        if col == numeric_field:
            continue
        if df[col].dtype == object:
            group_field = col
            break
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field after filtering and normalization. You can also plot relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered > {threshold})")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    # Normalized field
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
    plt.title(f"Normalized Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel("Frequency")
    plt.show()
    if group_field and group_field in filtered_df:
        plt.figure(figsize=(10, 5))
        # Show mean by group
        means = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
        sns.barplot(x=means.index, y=means.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(str(group_field))
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We have used the Croissant schema to examine structure, extract data by Croissant `@id`, and perform foundational EDA using the `mlcroissant` library. For more advanced analyses, refer to `mlcroissant` and dataset documentation.